In [1]:
import nbformat as nbf

nb = nbf.v4.new_notebook()
cells = []

# ── helper ──────────────────────────────────────────────────
def code(src): return nbf.v4.new_code_cell(src)
def md(src):   return nbf.v4.new_markdown_cell(src)

cells.append(md("# Ankh RAG — Egyptian Heritage Podcast Generator\n\nThis notebook implements a Retrieval-Augmented Generation (RAG) pipeline to create personalized podcast scripts about ancient Egyptian artifacts and sites. It uses a multimodal embedding model (DINOv2) for image-based retrieval, Qdrant as a vector database, and the Groq API for large language model (LLM) generation.\n\n## Features\n- **Multimodal RAG**: Search for information using images or text queries.\n- **Persona-Driven Generation**: Customize podcast scripts based on age group, gender, language, and knowledge level.\n- **Structured Podcast Scripting**: Generates multi-section podcast episodes with specific word count targets and narrative styles.\n- **Gradio UI**: Interactive interface for easy experimentation.\n\n## How it Works\n1. **Setup**: Installs necessary libraries and mounts Google Drive to load pre-computed embeddings.\n2. **Embeddings & Qdrant**: Loads image and text embeddings into an in-memory Qdrant vector database.\n3. **DINOv2 & Groq**: Initializes the DINOv2 model for image embedding and the Groq API client for LLM calls.\n4. **Persona Engine**: Dynamically builds a `PersonaConfig` based on user preferences.\n5. **Prompt Templates**: Constructs tailored prompts for each podcast section, incorporating retrieved context and persona details.\n6. **Podcast Generation**: Orchestrates the retrieval and generation process, calling the Groq LLM sequentially for each section.\n7. **Gradio Interface**: Provides a user-friendly web interface to interact with the RAG pipeline.\n"))

In [2]:
# ============================================================
# CELL 1 — Setup (unchanged, just add groq)
# ============================================================
cells.append(md("# Ankh RAG — Egypt Smart Tourism Assistant\n### Upgraded Podcast Script Generator (≤10 min)"))


# ============================================================
# CELL 1 — Setup & Install
# ============================================================
!pip install qdrant-client transformers accelerate gradio Pillow groq -q



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 9.0 MB/s eta 0:00:00


In [3]:
# ============================================================
# CELL 2 — Drive + Embeddings (unchanged)
# ============================================================


import pickle
from google.colab import drive
print("Connecting to Google Drive...")
drive.mount('/content/drive', force_remount=True)

Connecting to Google Drive...
Mounted at /content/drive


In [4]:

# ============================================================
# CELL 2 — Mount Google Drive & Load Pre-computed Embeddings
# ============================================================




PICKLE_PATH = "/content/drive/MyDrive/EgMM-Corpus/egmm_multimodal_embeddings.pkl"

print("Loading embeddings from pickle file...")
with open(PICKLE_PATH, "rb") as f:
    qdrant_points = pickle.load(f)

print(f"Done! Total points loaded: {len(qdrant_points)}")

Loading embeddings from pickle file...
Done! Total points loaded: 4654


In [5]:
# ============================================================
# CELL 3 — Qdrant (unchanged)
# ============================================================

# ============================================================
# CELL 3 — Build Qdrant In-Memory Vector Database
# ============================================================
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

qdrant_client = QdrantClient(":memory:")
COLLECTION_NAME = "egmm_corpus_multimodal"

print(f"Creating collection: '{COLLECTION_NAME}'...")
qdrant_client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "image_vector": VectorParams(size=768, distance=Distance.COSINE),
        "text_vector":  VectorParams(size=768, distance=Distance.COSINE),
    }
)

points_to_upload = [
    PointStruct(
        id=p["id"],
        vector={
            "image_vector": p["image_vector"],
            "text_vector":  p["text_vector"]
        },
        payload=p["payload"]
    )
    for p in qdrant_points
]

print(f"Uploading {len(points_to_upload)} points...")
qdrant_client.upsert(collection_name=COLLECTION_NAME, points=points_to_upload)
print("Qdrant DB is ready!")


Creating collection: 'egmm_corpus_multimodal'...
Uploading 4654 points...
Qdrant DB is ready!


In [6]:
# ============================================================
# CELL 4 — Load DINOv2 only (Qwen removed; LLM is Groq)
# ============================================================
# ============================================================
# CELL 4 — Load AI Models
# ============================================================
# DINOv2   : image → 768-dim embedding  (unchanged, runs locally)
# LLM      : Groq API  (llama-3.3-70b-versatile)
#            ─ no GPU memory needed for LLM
#            ─ much higher quality than Qwen-0.5B for long narrative
# ============================================================

import torch
from transformers import AutoImageProcessor, AutoModel
from groq import Groq
import os
from google.colab import userdata

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ── 4a: DINOv2 — Image Embedding Model (unchanged) ──
print("\\nLoading DINOv2 image encoder...")
DINO_MODEL_NAME = "facebook/dinov2-base"
img_processor = AutoImageProcessor.from_pretrained(DINO_MODEL_NAME)
img_model = AutoModel.from_pretrained(DINO_MODEL_NAME).to(device)
img_model.eval()
print("DINOv2 ready  (output dim: 768)")

# ── 4b: Groq API client ──
# Get your free key at https://console.groq.com
# Then set it here or via: import os; os.environ["GROQ_API_KEY"] = "gsk_..."
GROQ_API_KEY = userdata.get("Groq_API_KEY")
groq_client  = Groq(api_key=GROQ_API_KEY)

# Model options (all free on Groq):
#   "llama-3.3-70b-versatile"   ← best quality, use this
#   "llama-3.1-8b-instant"      ← fastest, lower quality
#   "mixtral-8x7b-32768"        ← large context, good alternative
GROQ_MODEL = "llama-3.3-70b-versatile"
print(f"\\nGroq LLM client ready  (model: {GROQ_MODEL})")


Using device: cpu
\nLoading DINOv2 image encoder...


preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

DINOv2 ready  (output dim: 768)
\nGroq LLM client ready  (model: llama-3.3-70b-versatile)


In [7]:
# ============================================================
# CELL 5 — Image Embedding Helper (unchanged)
# ============================================================

# ============================================================
# CELL 5 — Image → Embedding Helper  (unchanged)
# ============================================================
from PIL import Image
import torch

def get_image_embedding(image_input) -> list:
    if isinstance(image_input, str):
        image_input = Image.open(image_input)
    image_rgb = image_input.convert("RGB")
    inputs = img_processor(images=image_rgb, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = img_model(**inputs)
    cls_vector = outputs.last_hidden_state[:, 0, :]
    return cls_vector.cpu().numpy().flatten().tolist()

print("get_image_embedding() is ready!")


get_image_embedding() is ready!


In [8]:
# ============================================================
# CELL 6 — PersonaConfig dataclass
# ============================================================
cells.append(md("## Core Upgrade: Persona Engine + Podcast Script Generator"))
# ============================================================
# CELL 6 — Persona Engine
# ============================================================
# Converts (age_group, gender, language, knowledge_level)
# into a full PersonaConfig with derived narrative style.
#
# Rule-based — no model needed here.
# ============================================================

from dataclasses import dataclass
from typing import Literal

@dataclass
class PersonaConfig:
    age_group:       str   # "child" | "teen" | "adult" | "senior"
    gender:          str   # "male"  | "female" | "neutral"
    language:        str   # "en" | "ar" | "fr"
    knowledge_level: str   # "tourist" | "enthusiast" | "academic"
    # ── derived fields (set by build_persona) ──
    narrator_style:  str = ""   # "storyteller" | "professor" | "guide" | "explorer"
    vocab_level:     str = ""   # "simple" | "moderate" | "rich"
    tone:            str = ""   # "warm" | "dramatic" | "authoritative" | "playful"

# ── Lookup tables ──────────────────────────────────────────
_STYLE_MAP = {
    ("child",  "tourist"):     "storyteller",
    ("child",  "enthusiast"):  "storyteller",
    ("child",  "academic"):    "storyteller",
    ("teen",   "tourist"):     "explorer",
    ("teen",   "enthusiast"):  "explorer",
    ("teen",   "academic"):    "guide",
    ("adult",  "tourist"):     "guide",
    ("adult",  "enthusiast"):  "explorer",
    ("adult",  "academic"):    "professor",
    ("senior", "tourist"):     "guide",
    ("senior", "enthusiast"):  "professor",
    ("senior", "academic"):    "professor",
}

_VOCAB_MAP = {
    "child":  "simple",
    "teen":   "moderate",
    "adult":  "rich",
    "senior": "rich",
}

_TONE_MAP = {
    ("storyteller", "male"):    "playful",
    ("storyteller", "female"):  "warm",
    ("storyteller", "neutral"): "warm",
    ("explorer",    "male"):    "dramatic",
    ("explorer",    "female"):  "dramatic",
    ("explorer",    "neutral"): "dramatic",
    ("guide",       "male"):    "warm",
    ("guide",       "female"):  "warm",
    ("guide",       "neutral"): "warm",
    ("professor",   "male"):    "authoritative",
    ("professor",   "female"):  "authoritative",
    ("professor",   "neutral"): "authoritative",
}

def build_persona(
    age_group:       str = "adult",
    gender:          str = "male",
    language:        str = "en",
    knowledge_level: str = "tourist"
) -> PersonaConfig:
    """
    Factory function — constructs a PersonaConfig with all derived fields.

    Parameters
    ----------
    age_group       : "child" | "teen" | "adult" | "senior"
    gender          : "male"  | "female" | "neutral"
    language        : "en"    | "ar"    | "fr"
    knowledge_level : "tourist" | "enthusiast" | "academic"
    """
    style = _STYLE_MAP.get((age_group, knowledge_level), "guide")
    vocab = _VOCAB_MAP.get(age_group, "moderate")
    tone  = _TONE_MAP.get((style, gender), "warm")

    return PersonaConfig(
        age_group       = age_group,
        gender          = gender,
        language        = language,
        knowledge_level = knowledge_level,
        narrator_style  = style,
        vocab_level     = vocab,
        tone            = tone,
    )


# ── Quick test ──────────────────────────────────────────────
test_personas = [
    build_persona("child",  "male",    "en", "tourist"),
    build_persona("adult",  "female",  "en", "enthusiast"),
    build_persona("senior", "neutral", "ar", "academic"),
]
for p in test_personas:
    print(f"[{p.age_group}/{p.gender}/{p.knowledge_level}]  "
          f"→  style={p.narrator_style}  vocab={p.vocab_level}  tone={p.tone}")

print("\\nbuild_persona() is ready!")


[child/male/tourist]  →  style=storyteller  vocab=simple  tone=playful
[adult/female/enthusiast]  →  style=explorer  vocab=rich  tone=dramatic
[senior/neutral/academic]  →  style=professor  vocab=rich  tone=authoritative
\nbuild_persona() is ready!


In [9]:
# ============================================================
# CELL 7 — Prompt Templates
# ============================================================
# ============================================================
# CELL 7 — Prompt Template Builder
# ============================================================
# Each persona combination gets its OWN system prompt.
# Do NOT merge all personas into one prompt — the model
# will average them out and produce mediocre output for every
# persona instead of great output for the target audience.
#
# Word targets for ≤10 min podcast (@ 130 wpm):
#   intro_hook            :  130–180 words  (1 min)
#   historical_background :  380–450 words  (3 min)
#   construction_builders :  300–380 words  (2.5 min)
#   legends_stories       :  260–320 words  (2 min)
#   outro                 :  130–170 words  (1 min)
#   TOTAL                 : 1200–1500 words  (≤10 min)
# ============================================================

# ── Section structure (constant) ──────────────────────────
PODCAST_SECTIONS = [
    "intro_hook",
    "historical_background",
    "construction_builders",
    "legends_stories",
    "outro",
]

SECTION_WORD_TARGETS = {
    "intro_hook":            (130, 180),
    "historical_background": (380, 450),
    "construction_builders": (300, 380),
    "legends_stories":       (260, 320),
    "outro":                 (130, 170),
}

# ── Narrator persona descriptions ─────────────────────────
_NARRATOR_VOICE = {
    # (narrator_style, tone) → description for system prompt
    ("storyteller", "playful"): (
        "You are an enthusiastic storyteller narrating to children aged 7-12. "
        "Use short sentences. Replace all dates with vivid phrases like "
        "'thousands of years ago' or 'before your great-great-great-grandparents were born'. "
        "Use comparisons to everyday objects (like a school bus, a football field). "
        "Add 2-3 'wow' facts. Never use words with more than 3 syllables if a simpler word exists."
    ),
    ("storyteller", "warm"): (
        "You are a warm, nurturing storyteller narrating to girls aged 7-12. "
        "Focus on the beauty, colors, and human stories behind the artifact. "
        "Use gentle, vivid imagery. Short sentences. Replace dates with 'long, long ago'. "
        "Add small emotional moments — imagine the people who lived there."
    ),
    ("explorer", "dramatic"): (
        "You are an adventurous explorer-narrator speaking to teenagers aged 13-17. "
        "Use a documentary tone — think National Geographic. "
        "Include actual dates and names but frame them dramatically. "
        "Ask rhetorical questions. Build suspense before reveals."
    ),
    ("guide", "warm"): (
        "You are a knowledgeable, friendly Egyptian tour guide speaking to adult tourists. "
        "Balance historical facts with human interest. "
        "Use vivid scene-setting. Speak as if the listener is standing in front of the monument. "
        "Include dates, historical figures, and one surprising fact they won't find in guidebooks."
    ),
    ("professor", "authoritative"): (
        "You are a distinguished Egyptologist delivering an academic audio lecture. "
        "Use precise dates, architectural terminology, political context, and scholarly analysis. "
        "Reference the historical significance within the broader context of Egyptian history. "
        "Maintain narrative tension — this is a story, not a dry lecture. "
        "Speak to an educated adult who wants depth, not simplification."
    ),
}

def _get_narrator_voice(persona: PersonaConfig) -> str:
    key = (persona.narrator_style, persona.tone)
    # fallback to guide/warm if exact combo not found
    return _NARRATOR_VOICE.get(key, _NARRATOR_VOICE[("guide", "warm")])


# ── Section-specific instructions ─────────────────────────
_SECTION_INSTRUCTIONS = {
    "intro_hook": (
        "Open with a dramatic hook — a single vivid scene, sensory detail, or "
        "rhetorical question that places the listener at this site. "
        "Name the artifact/site explicitly. End the intro with a promise: "
        "'Today you will discover...' type transition."
    ),
    "historical_background": (
        "Tell the origin story. When was it built or discovered? "
        "What was the historical period? What empire, dynasty, or civilization? "
        "What was happening in Egypt and the world at that time? "
        "Make it feel like a story unfolding, not a list of facts."
    ),
    "construction_builders": (
        "Focus on WHO built it and HOW. Name the pharaoh, architect, or civilization. "
        "Describe the construction — scale, materials, engineering challenges. "
        "Include one concrete measurement or scale comparison if available in context. "
        "End with WHY they built it — purpose, religion, or power."
    ),
    "legends_stories": (
        "Share the myths, legends, and human stories connected to this site. "
        "What did ancient Egyptians believe about it? What mysteries still exist? "
        "Include any famous historical events that happened here. "
        "This section should feel like storytelling, not a history textbook."
    ),
    "outro": (
        "Bring the listener back to the present. "
        "What does this site mean today? What can visitors experience? "
        "End with a memorable closing thought, poetic reflection, or call to visit. "
        "Do NOT introduce new historical facts in the outro."
    ),
}


def build_section_prompt(
    section_name: str,
    persona: PersonaConfig,
    context_chunks: str,
    entity_name: str = "this ancient Egyptian site",
) -> str:
    """
    Builds the complete prompt for a single podcast section.

    Parameters
    ----------
    section_name  : one of PODCAST_SECTIONS
    persona       : PersonaConfig from build_persona()
    context_chunks: RAG-retrieved text, already joined
    entity_name   : resolved name of the artifact/site
    """
    word_min, word_max = SECTION_WORD_TARGETS[section_name]
    narrator_voice    = _get_narrator_voice(persona)
    section_task      = _SECTION_INSTRUCTIONS[section_name]

    prompt = f"""You are writing one section of a podcast script about {entity_name}.

== NARRATOR VOICE ==
{narrator_voice}

== YOUR TASK FOR THIS SECTION: {section_name.upper().replace("_", " ")} ==
{section_task}

== VERIFIED FACTUAL CONTEXT (from knowledge base) ==
Use ONLY these facts. Do NOT invent dates, names, measurements, or events
that are not present in the context below. If a fact is not here, do not include it.
If something is unknown, say "historians believe..." or "legend tells us...".

{context_chunks}

== OUTPUT FORMAT ==
- Write ONLY flowing prose. No bullet points. No headers. No section titles.
- Word count: {word_min}–{word_max} words (strictly enforced).
- This section will be read aloud — write for the ear, not the eye.
- Use natural spoken-word rhythm. Short sentences for drama. Longer ones for context.
- Language: {persona.language.upper()}
- Vocabulary level: {persona.vocab_level}

Write the {section_name.replace("_", " ")} section now:"""

    return prompt


print("Prompt template system ready!")
print(f"Sections: {PODCAST_SECTIONS}")
print(f"Total target: {sum(lo for lo,_ in SECTION_WORD_TARGETS.values())}–"
      f"{sum(hi for _,hi in SECTION_WORD_TARGETS.values())} words  (~≤10 min)")


Prompt template system ready!
Sections: ['intro_hook', 'historical_background', 'construction_builders', 'legends_stories', 'outro']
Total target: 1200–1500 words  (~≤10 min)


In [10]:
# ============================================================
# CELL 8 — Corrective RAG Retriever
# ============================================================

from enum import Enum

class RetrievalQuality(Enum):
    CORRECT   = "correct"
    AMBIGUOUS = "ambiguous"
    INCORRECT = "incorrect"

# ── Thresholds ────────────────────────────────────────────
HIGH_THRESHOLD       = 0.72
LOW_THRESHOLD        = 0.45
RECOGNITION_THRESHOLD = 0.55

# ──────────────────────────────────────────────────────────
# HELPER: Source tag for UI display
# ──────────────────────────────────────────────────────────
def _get_source_tag(quality_flag: str) -> str:
    """
    Returns a human-readable source label based on CRAG quality flag.
    Injected into both the prompt and the Gradio UI output box.
    """
    tags = {
        "correct":                "📚 [Source: Local knowledge base — high confidence retrieval]",
        "ambiguous":              "📚⚠️ [Source: Local knowledge base — some chunks filtered/refined]",
        "incorrect":              "🌐 [Source: Web search — local retrieval failed]",
        "unrecognized_recovered": "🌐 [Source: Web search — image not recognized, searched by name]",
        "unrecognized":           "❓ [Source: Unknown — image unrecognized, no name provided]",
    }
    return tags.get(quality_flag, "📚 [Source: Local knowledge base]")


# ──────────────────────────────────────────────────────────
# GATE 2 HELPERS: Chunk scoring and refinement
# ──────────────────────────────────────────────────────────
def score_chunk_relevance(
    chunk_text:  str,
    entity_name: str,
    groq_client,
    groq_model:  str,
) -> float:
    """
    LLM-as-judge: scores a single chunk's relevance to the entity.
    Returns float in [0.0, 1.0].
    Cost: ~150 input tokens + 5 output tokens per chunk.
    """
    judge_prompt = (
        f"You are a relevance judge. "
        f"Query entity: '{entity_name}'\n\n"
        f"Chunk:\n{chunk_text[:400]}\n\n"
        f"Is this chunk relevant to the query entity? "
        f"Reply with ONLY a number from 0.0 to 1.0. "
        f"1.0 = highly relevant. 0.0 = completely irrelevant. "
        f"No other text."
    )
    try:
        response = groq_client.chat.completions.create(
            model=groq_model,
            messages=[{"role": "user", "content": judge_prompt}],
            temperature=0.0,
            max_tokens=5,
        )
        raw = response.choices[0].message.content.strip()
        return max(0.0, min(1.0, float(raw)))
    except Exception:
        return 0.5   # default to ambiguous on parse/API failure


def refine_chunk(
    chunk_text:  str,
    entity_name: str,
    groq_client,
    groq_model:  str,
) -> str:
    """
    For AMBIGUOUS chunks: extracts only sentences relevant to the entity.
    Returns 'NO_RELEVANT_CONTENT' if nothing is worth keeping.
    """
    refine_prompt = (
        f"Entity: {entity_name}\n\n"
        f"Text:\n{chunk_text}\n\n"
        f"Extract ONLY the sentences that are directly relevant to {entity_name}. "
        f"Return them verbatim. If nothing is relevant, return 'NO_RELEVANT_CONTENT'."
    )
    try:
        response = groq_client.chat.completions.create(
            model=groq_model,
            messages=[{"role": "user", "content": refine_prompt}],
            temperature=0.0,
            max_tokens=300,
        )
        return response.choices[0].message.content.strip()
    except Exception:
        return chunk_text   # use raw chunk if refinement API call fails


# ──────────────────────────────────────────────────────────
# FALLBACK: Web search via Groq compound-beta-mini
# ──────────────────────────────────────────────────────────
def web_search_fallback(entity_name: str, groq_client, groq_model: str) -> str:
    """
    Gate 2 fallback: triggered when ALL retrieved chunks score below LOW_THRESHOLD.
    Uses Groq compound-beta-mini (Tavily-powered, no extra API key needed).
    Falls back to LLM self-knowledge if compound call also fails.
    """
    print(f"  ⚠ [CRAG Gate 2] Triggering web search fallback for: {entity_name}")

    search_prompt = (
        f"Search the web and find factual information about: {entity_name} "
        f"in the context of ancient Egypt. "
        f"Return ONLY confirmed facts — historical dates, who built it, "
        f"its purpose, and any notable features. "
        f"Format each fact on a new line starting with 'Fact: '. "
        f"Maximum 5 facts. Do not include opinions or uncertain claims."
    )
    try:
        response = groq_client.chat.completions.create(
            model="compound-beta-mini",
            messages=[{"role": "user", "content": search_prompt}],
            temperature=0.0,
            max_tokens=400,
        )
        result_text = response.choices[0].message.content.strip()
        print(f"  ✓ [CRAG Gate 2] Web fallback returned {len(result_text.split())} words")
        return result_text

    except Exception as e:
        print(f"  ✗ [CRAG Gate 2] Groq compound search failed: {e}")
        print(f"  ⚠ [CRAG Gate 2] Using LLM self-knowledge as last resort")
        try:
            last_resort = groq_client.chat.completions.create(
                model=groq_model,
                messages=[{
                    "role": "user",
                    "content": (
                        f"List up to 5 confirmed facts about {entity_name} "
                        f"in ancient Egypt. One per line, each starting with 'Fact: '. "
                        f"Only include facts you are certain about."
                    )
                }],
                temperature=0.0,
                max_tokens=300,
            )
            return last_resort.choices[0].message.content.strip()
        except Exception as e2:
            return f"Fact: {entity_name} is an ancient Egyptian monument. [All fallbacks failed: {e2}]"


# ──────────────────────────────────────────────────────────
# GATE 1: Image recognition check + web search by name
# ──────────────────────────────────────────────────────────
def check_image_recognized(
    query_vector: list,
    search_type:  str = "image_vector",
) -> tuple[bool, str, float]:
    """
    Checks if Qdrant's top result is a confident match.

    Returns
    -------
    is_recognized : bool  — True if top score >= RECOGNITION_THRESHOLD
    entity_name   : str   — name from top result payload
    top_score     : float — raw cosine similarity of top result
    """
    results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        using=search_type,
        limit=1,
        with_payload=True,
        score_threshold=0.0,
    ).points

    if not results:
        return False, "unknown monument", 0.0

    top = results[0]
    top_score   = top.score
    entity_name = (
        top.payload.get("Concept_Name")
        or top.payload.get("concept_name")
        or top.payload.get("name")
        or "unknown monument"
    )

    is_recognized = top_score >= RECOGNITION_THRESHOLD
    print(f"  [Gate 1] Top match: '{entity_name}'  score={top_score:.3f}  "
          f"→ {'RECOGNIZED ✓' if is_recognized else 'NOT RECOGNIZED ✗'}")

    return is_recognized, entity_name, top_score


def web_search_by_name(entity_name: str, groq_client, groq_model: str) -> str:
    """
    Gate 1 fallback: triggered when image is unrecognized but we have a name
    (from user_hint or weak Qdrant guess).
    Searches by known name — different from web_search_fallback() which
    recovers from bad chunks on a recognized entity.
    """
    print(f"  [Gate 1 fallback] Web searching by name: '{entity_name}'")

    search_prompt = (
        f"Search the web for detailed factual information about "
        f"'{entity_name}' as an ancient Egyptian monument or artifact. "
        f"Find: when it was built, who built it, its historical purpose, "
        f"its location in Egypt, and any notable features or legends. "
        f"Return facts only, one per line, each starting with 'Fact: '. "
        f"Maximum 6 facts."
    )
    try:
        response = groq_client.chat.completions.create(
            model="compound-beta-mini",
            messages=[{"role": "user", "content": search_prompt}],
            temperature=0.0,
            max_tokens=500,
        )
        result = response.choices[0].message.content.strip()
        print(f"  ✓ [Gate 1 fallback] Got {len(result.split())} words from web search")
        return result

    except Exception as e:
        print(f"  ✗ [Gate 1 fallback] Web search failed: {e}")
        return f"Fact: {entity_name} is an ancient Egyptian monument."

def identify_monument_from_image(image_input, groq_client, groq_model: str) -> str:
    """
    Uses Groq llama-4-scout vision LLM to identify the monument
    directly from the image when DINOv2/Qdrant fails to recognize it.
    Same Groq API key. No extra cost beyond token usage.
    Returns monument name as string, or "" if identification fails.
    """
    import base64
    from io import BytesIO
    from PIL import Image as PILImage

    print("  [Gate 1] Running vision LLM to identify monument from image...")

    try:
        if isinstance(image_input, str):
            image_input = PILImage.open(image_input)

        buffer = BytesIO()
        image_input.convert("RGB").save(buffer, format="JPEG")
        b64_image = base64.b64encode(buffer.getvalue()).decode("utf-8")

        response = groq_client.chat.completions.create(
            model="meta-llama/llama-4-scout-17b-16e-instruct",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{b64_image}"
                            }
                        },
                        {
                            "type": "text",
                            "text": (
                                "You are an expert Egyptologist. "
                                "Look at this image carefully. "
                                "If this is an Egyptian monument, artifact, temple, statue, "
                                "or historic site, respond with ONLY its name — nothing else. "
                                "Example responses: 'Karnak Temple', 'Sphinx of Giza', "
                                "'Bust of Nefertiti', 'Abu Simbel'. "
                                "If you cannot identify it or it is not Egyptian, "
                                "respond with exactly: UNKNOWN"
                            )
                        }
                    ]
                }
            ],
            temperature=0.0,
            max_tokens=30,
        )

        name = response.choices[0].message.content.strip()

        if name == "UNKNOWN" or not name:
            print("  [Gate 1] Vision LLM could not identify the monument")
            return ""

        print(f"  [Gate 1] Vision LLM identified: '{name}'")
        return name

    except Exception as e:
        print(f"  [Gate 1] Vision LLM failed: {e}")
        return ""


# ──────────────────────────────────────────────────────────
# MAIN: Full two-gate Corrective RAG retriever
# ──────────────────────────────────────────────────────────
def retrieve_context_corrective(
    query_vector: list,
    search_type:  str  = "image_vector",
    limit:        int  = 5,
    verbose:      bool = True,
    user_hint:    str  = None,
    image_input        = None,
) -> tuple[str, str, str]:
    """
    Full two-gate Corrective RAG retriever.

    Gate 1 — Was the image recognized by DINOv2/Qdrant?
    Gate 2 — Are the retrieved chunks actually relevant?

    Returns
    -------
    context_str  : refined context ready for the LLM
    entity_name  : resolved monument name
    quality_flag : 'correct' | 'ambiguous' | 'incorrect' |
                   'unrecognized' | 'unrecognized_recovered'
    """

    # ══════════════════════════════════════════════
    # GATE 1 — Image recognition check
    # ══════════════════════════════════════════════
    is_recognized, qdrant_entity, top_score = check_image_recognized(
        query_vector, search_type
    )

    if not is_recognized:
        # Priority 1: vision LLM identifies from image directly
        vision_name = ""
        if image_input is not None:
            vision_name = identify_monument_from_image(
                image_input, groq_client, GROQ_MODEL
            )

        if vision_name:
            entity_name = vision_name
            print(f"  [Gate 1] Using vision LLM identification: '{entity_name}'")

        elif user_hint and user_hint.strip():
            entity_name = user_hint.strip()
            print(f"  [Gate 1] Using user hint: '{entity_name}'")

        elif qdrant_entity != "unknown monument" and top_score > 0.35:
            entity_name = qdrant_entity
            print(f"  [Gate 1] Using weak Qdrant guess: '{entity_name}' (score={top_score:.3f})")

        else:
            print("  [Gate 1] All identification methods failed")
            return "UNRECOGNIZED_IMAGE", "unknown", "unrecognized"

        context_str = web_search_by_name(entity_name, groq_client, GROQ_MODEL)
        return context_str, entity_name, "unrecognized_recovered"
    # ══════════════════════════════════════════════
    # GATE 2 — Chunk relevance check
    # ══════════════════════════════════════════════
    results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        using=search_type,
        limit=limit,
    ).points

    entity_name = qdrant_entity

    raw_chunks = []
    seen = set()
    for r in results:
        chunk = r.payload.get("Text_Chunk") or r.payload.get("Full_Description") or ""
        chunk = chunk.strip()
        if chunk and chunk not in seen:
            seen.add(chunk)
            raw_chunks.append(chunk)

    if not raw_chunks:
        context_str = web_search_fallback(entity_name, groq_client, GROQ_MODEL)
        return context_str, entity_name, "incorrect"

    # Score every chunk
    scored_chunks = []
    for chunk in raw_chunks:
        score = score_chunk_relevance(chunk, entity_name, groq_client, GROQ_MODEL)
        scored_chunks.append((score, chunk))
        if verbose:
            print(f"  [Gate 2] Chunk score: {score:.2f} | {chunk[:60]}...")

    max_score = max(s for s, _ in scored_chunks)
    avg_score = sum(s for s, _ in scored_chunks) / len(scored_chunks)

    if max_score >= HIGH_THRESHOLD:
        overall_quality = RetrievalQuality.CORRECT
    elif avg_score <= LOW_THRESHOLD:
        overall_quality = RetrievalQuality.INCORRECT
    else:
        overall_quality = RetrievalQuality.AMBIGUOUS

    if verbose:
        print(f"  [Gate 2] max={max_score:.2f} avg={avg_score:.2f} → {overall_quality.value}")

    # Apply correction based on quality classification
    if overall_quality == RetrievalQuality.INCORRECT:
        context_str = web_search_fallback(entity_name, groq_client, GROQ_MODEL)

    elif overall_quality == RetrievalQuality.AMBIGUOUS:
        good_chunks = []
        for score, chunk in scored_chunks:
            if score >= HIGH_THRESHOLD:
                good_chunks.append(f"Fact: {chunk}")
            elif score > LOW_THRESHOLD:
                refined = refine_chunk(chunk, entity_name, groq_client, GROQ_MODEL)
                if refined != "NO_RELEVANT_CONTENT":
                    good_chunks.append(f"Fact (refined): {refined}")
            # score <= LOW_THRESHOLD → discard entirely
        context_str = "\n\n".join(good_chunks) if good_chunks else \
                      web_search_fallback(entity_name, groq_client, GROQ_MODEL)

    else:  # CORRECT
        good_chunks = [
            f"Fact: {chunk}"
            for score, chunk in scored_chunks
            if score >= LOW_THRESHOLD
        ]
        context_str = "\n\n".join(good_chunks)

    return context_str, entity_name, overall_quality.value


print("Cell 8 — Corrective RAG loaded successfully.")
print(f"Thresholds: RECOGNITION={RECOGNITION_THRESHOLD}  HIGH={HIGH_THRESHOLD}  LOW={LOW_THRESHOLD}")

Cell 8 — Corrective RAG loaded successfully.
Thresholds: RECOGNITION=0.55  HIGH=0.72  LOW=0.45


In [11]:
# ============================================================
# CELL 9 — Groq LLM caller
# ============================================================
# ============================================================
# CELL 9 — Groq LLM Caller
# ============================================================
# Sends a prompt to Groq and returns the generated text.
# Includes:
#   - temperature tuned per section type
#   - word-count post-validation (warns but does not crash)
#   - hallucination guard reminder in every system message
# ============================================================

def _section_temperature(section_name: str) -> float:
    """
    Creative sections get higher temperature.
    Factual sections get lower temperature.
    """
    return {
        "intro_hook":            0.75,
        "historical_background": 0.35,
        "construction_builders": 0.35,
        "legends_stories":       0.70,
        "outro":                 0.65,
    }.get(section_name, 0.5)


HALLUCINATION_GUARD_SYSTEM = (
    "CRITICAL RULE: You must ONLY use facts that appear in the provided context. "
    "If a date, name, or measurement is not in the context, do NOT include it. "
    "If something is uncertain, use hedging language: 'historians believe...', "
    "'according to ancient records...', 'legend suggests...'. "
    "Never invent specific numbers, proper names, or historical events."
)


def call_groq(
    prompt:       str,
    section_name: str,
    max_tokens:   int = 700,
) -> str:
    """
    Call Groq API and return the generated text.

    Parameters
    ----------
    prompt       : full section prompt from build_section_prompt()
    section_name : used to set temperature
    max_tokens   : hard cap; 700 ≈ 500 words (safe margin above section targets)
    """
    try:
        response = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {"role": "system", "content": HALLUCINATION_GUARD_SYSTEM},
                {"role": "user",   "content": prompt},
            ],
            temperature=_section_temperature(section_name),
            max_tokens=max_tokens,
        )
        generated = response.choices[0].message.content.strip()

        # ── Post-generation word count check ──
        word_count = len(generated.split())
        w_min, w_max = SECTION_WORD_TARGETS[section_name]
        if word_count < w_min:
            print(f"  ⚠ [{section_name}] Generated {word_count} words — below target ({w_min})")
        elif word_count > w_max + 50:   # allow 50-word overflow
            print(f"  ⚠ [{section_name}] Generated {word_count} words — above target ({w_max})")
        else:
            print(f"  ✓ [{section_name}] {word_count} words  (target: {w_min}–{w_max})")

        return generated

    except Exception as e:
        print(f"  ✗ [{section_name}] Groq API error: {e}")
        return f"[Generation failed for {section_name}: {e}]"


print("call_groq() is ready!")


call_groq() is ready!


In [12]:
# ============================================================
# CELL 10 — Full podcast generator
# ============================================================
# ============================================================
# CELL 10 — Full Podcast Script Generator
# ============================================================
# Orchestrates the complete pipeline:
#
#   image / text query
#       ↓
#   DINOv2 embedding  (image path)  OR  text embedding (text path)
#       ↓
#   Qdrant retrieval  →  context_str + entity_name
#       ↓
#   PersonaConfig  →  5 section prompts
#       ↓
#   Groq LLM  (one API call per section, sequential)
#       ↓
#   PodcastScript  (validated, assembled)
# ============================================================
import time
from dataclasses import dataclass, field

GROQ_TPM_LIMIT    = 12000   # free tier for llama-3.3-70b
GROQ_SAFETY_RATIO = 0.80    # stay at 80% of limit to avoid 429s
TOKENS_PER_SECTION_APPROX = 1400  # 800 input + 600 output estimate

def safe_sleep_between_calls(tokens_just_used: int = TOKENS_PER_SECTION_APPROX):
    """
    Adaptive sleep: waits longer when token usage is high.
    At 12K TPM, we can send 200 tokens/sec safely.
    A 1400-token call needs ~7 seconds of budget recovery.
    """
    recovery_seconds = tokens_just_used / (GROQ_TPM_LIMIT * GROQ_SAFETY_RATIO / 60)
    sleep_time = max(2.0, recovery_seconds)   # minimum 2s between calls
    time.sleep(sleep_time)

@dataclass
class PodcastScript:
    entity_name:        str
    persona:            PersonaConfig
    sections:           dict = field(default_factory=dict)  # section_name → text
    total_word_count:   int  = 0
    estimated_duration: float = 0.0   # minutes
    retrieval_quality:  str  = ""     # Add this line to the dataclass

    def assemble(self) -> str:
        """Return the full script as a single string with section markers."""
        parts = []
        for section in PODCAST_SECTIONS:
            text = self.sections.get(section, "")
            if text:
                parts.append(f"--- {section.upper().replace('_', ' ')} ---\n{text}")
        return "\n\n".join(parts)


def generate_podcast_script(
    image_input      = None,       # PIL.Image or str path (for image query)
    text_query       : str = None, # text query if no image
    age_group        : str = "adult",
    gender           : str = "male",
    language         : str = "en",
    knowledge_level  : str = "tourist",
    rag_limit        : int = 5,
    verbose          : bool = True,
    user_hint        : str = None,
) -> PodcastScript:
    """
    Main entry point. Provide EITHER image_input OR text_query.

    Parameters
    ----------
    image_input    : PIL.Image or str file path
    text_query     : string query (used if no image provided)
    age_group      : "child" | "teen" | "adult" | "senior"
    gender         : "male"  | "female" | "neutral"
    language       : "en"    | "ar"    | "fr"
    knowledge_level: "tourist" | "enthusiast" | "academic"
    rag_limit      : number of Qdrant results to retrieve (5 recommended)
    verbose        : print progress
    """
    print("=" * 60)
    print("ANKH PODCAST GENERATOR — starting pipeline")
    print("=" * 60)

    # ── Step 1: Build PersonaConfig ─────────────────
    persona = build_persona(age_group, gender, language, knowledge_level)
    if verbose:
        print(f"\n[Persona] style={persona.narrator_style}  "
              f"vocab={persona.vocab_level}  tone={persona.tone}")

    # ── Step 2: Generate query vector ─────────────────
    if image_input is not None:
        if verbose: print("\n[Step 2] Embedding image via DINOv2...")
        query_vector = get_image_embedding(image_input)
        search_type  = "image_vector"
    elif text_query:
        # For text queries we reuse the stored text_vectors via a simple
        # keyword match on payload — a proper text embedding model would
        # be better, but this works for the current dataset.
        # Quick workaround: find the point whose Concept_Name best matches.
        if verbose: print(f"\n[Step 2] Text query mode: '{text_query}'")
        # Use the first matching point\'s text_vector as the query
        matched = [
            p for p in qdrant_points
            if text_query.lower() in str(p["payload"].get("Concept_Name","")).lower()
            or text_query.lower() in str(p["payload"].get("Text_Chunk","")).lower()
        ]
        if matched:
            query_vector = matched[0]["text_vector"]
            if verbose: print(f"   Found match: {matched[0]['payload'].get('Concept_Name','?')}")
        else:
            # Fall back to first point in DB (graceful degradation)
            query_vector = qdrant_points[0]["text_vector"]
            if verbose: print("   No exact match — using first DB entry as fallback")
        search_type = "text_vector"
    else:
        raise ValueError("Provide either image_input or text_query")

    # ── Step 3: Retrieve context from Qdrant ───────────
    if verbose: print("\n[Step 3] Retrieving context from Qdrant...")
    # NEW — CRAG version:
    context_str, entity_name, retrieval_quality = retrieve_context_corrective(
        query_vector, search_type, rag_limit, verbose=verbose, user_hint=user_hint, image_input=image_input,
    )
    if verbose:
        print(f"   Quality flag : {retrieval_quality}")
    # script.retrieval_quality = retrieval_quality.value   # This line is now redundant as it's passed in constructor
    if verbose:
        print(f"   Entity : {entity_name}")
        print(f"   Context: {len(context_str.split())} words retrieved")

    # ── Step 4: Generate each section via Groq ──────────────
    if verbose: print("\n[Step 4] Generating podcast sections via Groq LLM...")
    # Fix: Pass the value of retrieval_quality correctly and initialize the dataclass
    script = PodcastScript(entity_name=entity_name, persona=persona, retrieval_quality=retrieval_quality)

    for section in PODCAST_SECTIONS:
        if verbose: print(f"\n  Generating: {section}...")
        t0 = time.time()

        prompt = build_section_prompt(
            section_name  = section,
            persona       = persona,
            context_chunks= context_str,
            entity_name   = entity_name,
        )
        text = call_groq(prompt, section_name=section)
        script.sections[section] = text

        elapsed = time.time() - t0
        if verbose: print(f"     ({elapsed:.1f}s)")

        # Small delay to avoid Groq rate limit (free tier: 30 req/min)
        safe_sleep_between_calls()

    # ── Step 5: Compute stats ─────────────────
    full_text = " ".join(script.sections.values())
    script.total_word_count   = len(full_text.split())
    script.estimated_duration = script.total_word_count / 130   # minutes @ 130 wpm

    print("\n" + "=" * 60)
    print(f"DONE  |  {script.total_word_count} words  "
          f"|  ~{script.estimated_duration:.1f} min estimated")
    print("=" * 60)

    return script


print("generate_podcast_script() is ready!")

generate_podcast_script() is ready!


In [13]:
# ============================================================
# CELL 11 — Legacy wrapper (backward compatibility)
# ============================================================
# ============================================================
# CELL 11 — Legacy RAG Functions (Backward Compatibility)
# ============================================================
# These keep the original get_strict_summary() and
# get_personalized_story() working so old Gradio cells don\'t break.
#
# BUT: the recommended way is now generate_podcast_script().
# ============================================================

from IPython.display import display, Markdown

def get_strict_summary(query_vector: list, search_type: str = "image_vector", limit: int = 3):
    """Legacy: academic description (short, not a podcast)."""
    context_str, entity_name, quality_flag = retrieve_context_corrective(
    query_vector, search_type, limit, verbose=False
    )
    if context_str == "UNRECOGNIZED_IMAGE":
        return "⚠️ Image not recognized. Please type the monument name and try again.", ""
    prompt = (
        f"You are an expert Egyptologist Tour Guide. "
        f"Provide a comprehensive, structured explanation of {entity_name} "
        f"using ONLY the facts below.\\n\\n"
        f"{context_str}\\n\\n"
        f"Write clear, readable paragraphs. English only. ~200 words."
    )
    try:
        response = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {"role": "system", "content": HALLUCINATION_GUARD_SYSTEM},
                {"role": "user",   "content": prompt},
            ],
            temperature=0.3,
            max_tokens=400,
        )
        return response.choices[0].message.content.strip(), context_str
    except Exception as e:
        return f"Generation failed: {e}", context_str


def get_personalized_story(
    query_vector: list,
    search_type:  str = "image_vector",
    user_persona: str = "man_adult",
    limit:        int = 3
):
    """
    Legacy: short personalized story (2 paragraphs).
    Keeps old persona keys: boy_child | girl_child | man_adult | woman_adult
    """
    # Map old keys to new PersonaConfig
    _LEGACY_MAP = {
        "boy_child":   ("child",  "male"),
        "girl_child":  ("child",  "female"),
        "man_adult":   ("adult",  "male"),
        "woman_adult": ("adult",  "female"),
    }
    age, gender = _LEGACY_MAP.get(user_persona, ("adult", "male"))
    persona     = build_persona(age, gender, "en", "tourist")

    context_str, entity_name, _ = retrieve_context_corrective(query_vector, search_type, limit)

    narrator = _get_narrator_voice(persona)
    prompt = (
        f"{narrator}\\n\\n"
        f"Write exactly 2 short paragraphs (2-3 sentences each) about {entity_name}. "
        f"Use ONLY these facts:\\n{context_str}\\n\\n"
        f"No bullet points. No headers. Flowing prose. English only."
    )
    try:
        response = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {"role": "system", "content": HALLUCINATION_GUARD_SYSTEM},
                {"role": "user",   "content": prompt},
            ],
            temperature=0.6,
            max_tokens=350,
        )
        return response.choices[0].message.content.strip(), context_str
    except Exception as e:
        return f"Story generation failed: {e}", context_str


print("Legacy functions (get_strict_summary + get_personalized_story) ready!")
print("Recommended: use generate_podcast_script() for full podcast output.")


Legacy functions (get_strict_summary + get_personalized_story) ready!
Recommended: use generate_podcast_script() for full podcast output.


In [14]:
# ============================================================
# CELL 12 — Pipeline wrapper
# ============================================================
# ============================================================
# CELL 12 — ankh_rag_pipeline  (updated to support podcast mode)
# ============================================================

def ankh_rag_pipeline(image, mode="description", persona="man_adult", user_hint=None):
    """
    Handles description and story modes.
    Now uses full two-gate CRAG retrieval.
    """
    # Step 1: embed image
    query_vector = get_image_embedding(image)

    # Step 2: CRAG retrieval (same as podcast mode)
    context_str, entity_name, quality_flag = retrieve_context_corrective(
        query_vector,
        search_type="image_vector",
        limit=5,
        verbose=True,
        user_hint=user_hint,
        image_input=image,
    )

    # Step 3: handle unrecognized image
    if context_str == "UNRECOGNIZED_IMAGE":
        return (
            "⚠️ I could not identify this monument from the image alone.\n\n"
            "Please type the monument name in the 'Monument Name (optional)' "
            "field and click Generate again.",
            "Image not recognized — Gate 1 failed, no name available"
        )

    # Step 4: tag context source for transparency
    source_tag = _get_source_tag(quality_flag)

    # Step 5: build prompt based on mode
    _LEGACY_MAP = {
      "boy_child":   ("child",  "male"),
      "girl_child":  ("child",  "female"),
      "man_adult":   ("adult",  "male"),
      "woman_adult": ("adult",  "female"),
    }
    age, gen = _LEGACY_MAP.get(persona, ("adult", "male"))
    persona_config = build_persona(age, gen, "en", "tourist")

    if mode == "description":
        prompt = f"""You are an Egyptian heritage guide.
{source_tag}

Context:
{context_str}

Describe {entity_name} in detail. Cover its history, construction, purpose, and significance.
Write in a clear, informative style suitable for a general audience.
STRICT RULE: Only use facts from the context above. Do not invent details."""

    elif mode == "story":
        prompt = f"""You are a storyteller specializing in ancient Egypt.
{source_tag}

Context:
{context_str}

Tell an engaging short narrative about {entity_name}.
Make it vivid and immersive, but grounded only in the facts provided above.
STRICT RULE: Do not invent facts not present in the context."""

    else:
        prompt = f"Describe {entity_name} based on: {context_str}"

    # Step 6: generate
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": HALLUCINATION_GUARD_SYSTEM},
            {"role": "user",   "content": prompt}
        ],
        temperature=0.7,
        max_tokens=800,
    )

    output_text = response.choices[0].message.content.strip()

    # Step 7: append source notice to output
    context_info = f"Entity: {entity_name} | Source: {source_tag} | Quality: {quality_flag}"

    return output_text, context_info


In [15]:
# ============================================================
# CELL 13 — Gradio UI (updated)
# ============================================================
# ============================================================
# CELL 13 — Gradio Interactive Interface (Updated)
# ============================================================

import gradio as gr
from PIL import Image

def gradio_handler(image, mode, persona, age_group, gender, language, knowledge_level, monument_hint):
    if image is None:
        return "Please upload an image first.", "", "—"

    try:

        # ── Route to correct mode ─────────────────────────────
        if mode == "podcast":
            script = generate_podcast_script(
                image_input     = image,
                age_group       = age_group,
                gender          = gender,
                language        = language,
                knowledge_level = knowledge_level,
                user_hint       = monument_hint.strip() or None,
            )
            assembled    = script.assemble()
            source_tag   = _get_source_tag(script.retrieval_quality)
            context_info = (
                f"Entity: {script.entity_name} | "
                f"{script.total_word_count} words | "
                f"~{script.estimated_duration:.1f} min | "
                f"{source_tag}"
            )
            persona_info = f"Persona: {script.persona.narrator_style} / {script.persona.tone} / vocab={script.persona.vocab_level}"
            return assembled, context_info, persona_info

        else:
            # description or story — now uses CRAG
            response, context_info = ankh_rag_pipeline(
                image,
                mode      = mode,
                persona   = persona,
                user_hint = monument_hint.strip() or None,
            )
            source_tag   = context_info   # already formatted inside ankh_rag_pipeline
            persona_info = f"Mode: {mode} | Persona: {persona} | DB points: {len(qdrant_points)}"
            return response, context_info, persona_info

    except Exception as e:
        err = f"Error: {str(e)}"
        return err, err, "—"


with gr.Blocks(title="Ankh RAG — Egyptian Heritage Guide", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🏛️ Ankh RAG — Egyptian Heritage Podcast Generator
    Upload an image of an Egyptian artifact or historic site.
    Choose **Podcast** mode for a full narrated ~10-minute script.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            img_input = gr.Image(type="pil", label="Upload Artifact / Site Image")

            mode_radio = gr.Radio(
                choices=["description", "story", "podcast"],
                value="description",
                label="Response Mode",
                info="'description'=academic | 'story'=short narrative | 'podcast'=full ~10min script"
            )

            gr.Markdown("### Quick Persona (for description/story modes)")
            persona_dropdown = gr.Dropdown(
                choices=["man_adult", "woman_adult", "boy_child", "girl_child"],
                value="man_adult",
                label="Quick Persona"
            )

            gr.Markdown("### Full Persona Controls (for podcast mode)")
            age_dd  = gr.Dropdown(["child","teen","adult","senior"], value="adult", label="Age Group")
            gen_dd  = gr.Dropdown(["male","female","neutral"], value="male", label="Gender")
            lang_dd = gr.Dropdown(["en","ar","fr"], value="en", label="Language")
            kl_dd   = gr.Dropdown(["tourist","enthusiast","academic"], value="tourist", label="Knowledge Level")

            monument_hint = gr.Textbox(
                label="Monument Name (optional)",
                placeholder="e.g. 'Karnak Temple' — fill this if the image is unclear",
                value=""
            )

            submit_btn = gr.Button("Generate 🔍", variant="primary")

        with gr.Column(scale=1):
            output_response = gr.Textbox(label="Generated Output", lines=25)
            output_context  = gr.Textbox(label="Context / Stats", lines=4)
            output_info     = gr.Textbox(label="Persona Info", lines=2, interactive=False)

    submit_btn.click(
        fn=gradio_handler,
        inputs=[img_input, mode_radio, persona_dropdown, age_dd, gen_dd, lang_dd, kl_dd, monument_hint],
        outputs=[output_response, output_context, output_info]
    )

    gr.Markdown("""
    ---
    **Pipeline:** Image → DINOv2 embedding → Qdrant retrieval → PersonaConfig → 5-section Groq LLM → Podcast script
    """)

print("Launching Gradio...")
demo.launch(debug=True, share=True)


/tmp/ipykernel_3999/3334146123.py:55: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Ankh RAG — Egyptian Heritage Guide", theme=gr.themes.Soft()) as demo:


Launching Gradio...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://34097504dea4bb1716.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


  [Gate 1] Top match: 'Nubia Museum, Aswan'  score=0.728  → RECOGNIZED ✓
  [Gate 2] Chunk score: 0.80 | # Wikipedia: Aswan Museum
Aswan Museum is a museum in Elepha...
  [Gate 2] Chunk score: 0.80 | # Wikipedia: Aswan Museum
Aswan Museum is a museum in Elepha...
  [Gate 2] Chunk score: 0.00 | # Wikipedia: Imhotep Museum
The Imhotep Museum is an archaeo...
  [Gate 2] Chunk score: 0.00 | # Wikipedia: White Pyramid
The White Pyramid (Egyptian Arabi...
  [Gate 2] max=0.80 avg=0.40 → correct
  [Gate 1] Top match: 'White Pyramid of Amenemhat II'  score=0.745  → RECOGNIZED ✓
  [Gate 2] Chunk score: 1.00 | # Wikipedia: White Pyramid
The White Pyramid (Egyptian Arabi...
  [Gate 2] Chunk score: 0.00 | # Wikipedia: Sphinx of Memphis
The Sphinx of Memphis or Alab...
  [Gate 2] Chunk score: 0.00 | # Wikipedia: Pyramid of Ahmose
The pyramid of Ahmose is a py...
  [Gate 2] Chunk score: 0.00 | # Wikipedia: Al-Gawhara Palace
Al-Gawhara Palace (Arabic: قص...
  [Gate 2] max=1.00 avg=0.25 → correct
  [Gat

In [16]:
# ============================================================
# CELL 14 — Direct test (no Gradio)
# ============================================================

# ============================================================
# CELL 14 — Direct Test: generate a podcast from a stored vector
# ============================================================
# This tests the full pipeline without needing a real image upload.
# Uses the first stored embedding from the DB as the query vector.
# ============================================================

from PIL import Image as PILImage
import numpy as np

# Simulate an image query using the first stored image_vector
# (In real use, this would be a user-uploaded image)
test_vector = qdrant_points[0]["image_vector"]

# Manually call retrieve_context to see what entity we\'re working with
# REPLACE with:
test_context, test_entity, test_quality = retrieve_context_corrective(
    test_vector, "image_vector", limit=5, verbose=True
)
print(f"Retrieval quality: {test_quality}")

# Build a persona
test_persona = build_persona("adult", "female", "en", "enthusiast")
print(f"Persona: {test_persona}")
print()

# Generate one section to verify everything works before running all 5
print("--- Testing single section (intro_hook) ---")
test_prompt = build_section_prompt(
    section_name  = "intro_hook",
    persona       = test_persona,
    context_chunks= test_context,
    entity_name   = test_entity,
)
test_output = call_groq(test_prompt, section_name="intro_hook")
print()
print(test_output)


  [Gate 1] Top match: 'Bab al-Nasr (Cairo)'  score=1.000  → RECOGNIZED ✓
  [Gate 2] Chunk score: 1.00 | # Wikipedia: Bab al-Nasr (Cairo)
Bab al-Nasr (Arabic: باب ال...
  [Gate 2] Chunk score: 0.00 | # Wikipedia: Mosque-Sabil of Sulayman Agha al-Silahdar
The M...
  [Gate 2] Chunk score: 0.00 | # Wikipedia: Gayer-Anderson Museum
The Gayer-Anderson Museum...
  [Gate 2] max=1.00 avg=0.33 → correct
Retrieval quality: correct
Persona: PersonaConfig(age_group='adult', gender='female', language='en', knowledge_level='enthusiast', narrator_style='explorer', vocab_level='rich', tone='dramatic')

--- Testing single section (intro_hook) ---
  ⚠ [intro_hook] Generated 112 words — below target (130)

Imagine standing before the majestic Bab al-Nasr, its semicircular arch a testament to a rich history. What secrets lie behind this Gate of Victory, a name that has endured for centuries? Historians believe it was once known as the Gate of Prosperity, but the people preferred a different name. As you ap

In [17]:
cells.append(code('''\
# ============================================================
# CELL 15 — Full Podcast Generation Test
# ============================================================
# Run this only after Cell 14 succeeds.
# Generates all 5 sections (~5 Groq API calls, ~15-20 seconds).
# ============================================================

# Option A: generate from a stored vector (no image needed for testing)
# We create a tiny 1x1 white image and manually override with stored vector
# by calling generate_podcast_script with a text query instead

full_script = generate_podcast_script(
    text_query      = qdrant_points[0]["payload"].get("Concept_Name", "egypt"),
    age_group       = "adult",
    gender          = "female",
    language        = "en",
    knowledge_level = "enthusiast",
    verbose         = True,
)

print("\\n\\n" + "=" * 60)
print("FULL ASSEMBLED SCRIPT")
print("=" * 60)
print(full_script.assemble())
'''))
